In [3]:
def bytes_per_element(dtype: str) -> int:
  d = dtype.lower()
  if d in ("fp32", "float32"):
      return 4
  if d in ("bf16", "bfloat16", "fp16", "float16"):
      return 2
  if d in ("fp8", "int8", "uint8"):
      return 1
  raise ValueError("Unknown dtype: {}".format(dtype))

def gb(num_bytes: float) -> float:
  # Decimal GB (1e9 bytes) keeps the arithmetic simple.
  return num_bytes / 1e9


In [6]:

def print_breakdown(title: str, b: dict):
  print(title)
  print("  Weights:          {:>7} GB".format(b['weights_gb']))
  print("  KV cache:         {:>7} GB".format(b['kv_cache_gb']))
  print("  Activations:      {:>7} GB".format(b['activations_gb']))
  print("  Grads:            {:>7} GB".format(b['grads_gb']))
  print("  Optimizer states: {:>7} GB".format(b['optimizer_states_gb']))
  if b["fp32_master_weights_gb"] > 0:
      print("  FP32 master:      {:>7} GB".format(b['fp32_master_weights_gb']))
  print("  -> Inference total (w/ overhead): {} GB".format(b['inference_total_gb']))
  print("  -> Training total  (w/ overhead): {} GB".format(b['training_total_gb']))
  print()


In [16]:

def estimate_transformer_memory_gb(
  params_billions: float,
  batch_size: int = 1,
  seq_length: int = 2048,
  hidden_dim: int = 4096,
  num_layers: int = 32,
  *,
  param_dtype: str = "fp32",
  grad_dtype: str = "fp32",
  optimizer: str = "adam",
  fp32_master_weights: bool = True,
  kv_dtype=None,
  kv_head_ratio: float = 1.0,  # 1.0 = MHA, <1.0 = GQA/MQA
  activation_multiplier: float = 4.0,  # rough saved tensors per token per layer
  activation_checkpointing_factor: float = 1.0,  # ~0.25 is a common ballpark
  overhead_frac: float = 0.15,  # fragmentation + temp buffers + kernels
) -> dict:
  """Back-of-the-envelope VRAM estimate (decimal GB).

  Notes:
  - Optimizer states are assumed FP32 (common default).
  - Activations are intentionally a rough multiplier; the point is correct scaling.
  """

  params = params_billions * 1e9
  param_b = bytes_per_element(param_dtype)
  grad_b = bytes_per_element(grad_dtype)
  kv_b = bytes_per_element(kv_dtype or param_dtype)

  # Capacity terms
  weights_bytes = params * param_b

  # KV cache (inference): K and V per layer
  kv_cache_bytes = (
      2 * batch_size * seq_length * hidden_dim * kv_b * num_layers * kv_head_ratio
  )

  # Training-only terms
  grads_bytes = params * grad_b

  master_weights_bytes = 0
  if fp32_master_weights and param_b < 4:
      # Many mixed-precision setups keep an FP32 copy of weights.
      master_weights_bytes = params * 4

  optimizer_bytes = 0
  opt = optimizer.lower()
  if opt == "adam":
      # Adam keeps m and v (often FP32).
      optimizer_bytes = params * 2 * 4
  elif opt == "sgd":
      # SGD with momentum keeps 1 velocity buffer (FP32).
      optimizer_bytes = params * 1 * 4
  elif opt == "none":
      optimizer_bytes = 0
  else:
      raise ValueError("optimizer must be 'adam' or 'sgd'")

  # Activations (training): scales with tokens and depth.
  activations_bytes = (
      batch_size
      * seq_length
      * hidden_dim
      * num_layers
      * activation_multiplier
      * param_b
      * activation_checkpointing_factor
  )

  inference_total = weights_bytes + kv_cache_bytes
  training_total = (
      weights_bytes
      + grads_bytes
      + master_weights_bytes
      + optimizer_bytes
      + activations_bytes
  )

  # Add a simple overhead budget to each total.
  inference_total *= 1 + overhead_frac
  training_total *= 1 + overhead_frac

  return {
      "weights_gb": round(gb(weights_bytes), 2),
      "kv_cache_gb": round(gb(kv_cache_bytes), 2),
      "activations_gb": round(gb(activations_bytes), 2),
      "grads_gb": round(gb(grads_bytes), 2),
      "optimizer_states_gb": round(gb(optimizer_bytes), 2),
      "fp32_master_weights_gb": round(gb(master_weights_bytes), 2),
      "inference_total_gb": round(gb(inference_total), 2),
      "training_total_gb": round(gb(training_total), 2),
  }

In [23]:
model_configs = [
    ("7B",   7,   4096, 32),   # Llama-7B-class
  ("7B",   7,   8192, 32),   # Llama-7B-class
  ("70B",  70,  8192, 80),   # Llama-70B-class
  ("175B", 175, 12288, 96),  # GPT-3-175B-class
]

for name, params, hdim, nlayers in model_configs:
  b = estimate_transformer_memory_gb(
      params,
      batch_size=1,
      seq_length=2048,
      hidden_dim=hdim,
      num_layers=nlayers,
      param_dtype="bf16",
      optimizer="adam",
      overhead_frac=0.15,
  )
  print_breakdown("=== {} (h={}, L={}) @ seq=2048 (fp32 params) ===".format(name, hdim, nlayers), b)


# Now feel the KV-cache wall: increase seq_length and watch KV cache grow.
b = estimate_transformer_memory_gb(7, seq_length=128_000, kv_head_ratio=1.0, overhead_frac=0.15)
print_breakdown("=== 7B @ seq=128k (KV cache dominates) ===", b)
b = estimate_transformer_memory_gb(7, seq_length=64_000, kv_head_ratio=1.0, overhead_frac=0.15)
print_breakdown("=== 7B @ seq=64k (KV cache dominates) ===", b)


=== 7B (h=4096, L=32) @ seq=2048 (fp32 params) ===
  Weights:             14.0 GB
  KV cache:            1.07 GB
  Activations:         2.15 GB
  Grads:               28.0 GB
  Optimizer states:    56.0 GB
  FP32 master:         28.0 GB
  -> Inference total (w/ overhead): 17.33 GB
  -> Training total  (w/ overhead): 147.37 GB

=== 7B (h=8192, L=32) @ seq=2048 (fp32 params) ===
  Weights:             14.0 GB
  KV cache:            2.15 GB
  Activations:         4.29 GB
  Grads:               28.0 GB
  Optimizer states:    56.0 GB
  FP32 master:         28.0 GB
  -> Inference total (w/ overhead): 18.57 GB
  -> Training total  (w/ overhead): 149.84 GB

=== 70B (h=8192, L=80) @ seq=2048 (fp32 params) ===
  Weights:            140.0 GB
  KV cache:            5.37 GB
  Activations:        10.74 GB
  Grads:              280.0 GB
  Optimizer states:   560.0 GB
  FP32 master:        280.0 GB
  -> Inference total (w/ overhead): 167.17 GB
  -> Training total  (w/ overhead): 1461.35 GB

=== 175B (

In [21]:
def gb(num_bytes: float) -> float:
  return num_bytes / 1e9


# Same shapes as the PyTorch example above
batch, seq, hidden = 32, 6000, 16384
dtype_bytes = 4  # float32

x = batch * seq * hidden * dtype_bytes          # input
y = batch * seq * hidden * dtype_bytes          # output
w = hidden * hidden * dtype_bytes               # weights

print("Input x:   {:.2f} GB".format(gb(x)))
print("Output y:  {:.2f} GB".format(gb(y)))
print("Weights W: {:.2f} GB".format(gb(w)))

# A common mistake is to count only forward tensors.
forward_only = gb(x + y + w)
print("\nForward-only (optimistic): {:.2f} GB".format(forward_only))

# Backward adds gradients. At peak, you often have:
# - x, y, W still resident
# - grad_output (same size as y)
# - grad_input  (same size as x)
# - grad_weight (same size as W)
backward_tensors = gb(x + y + w + y + x + w)
print("Backward tensors (still optimistic): {:.2f} GB".format(backward_tensors))

# Real runs pay extra: allocator fragmentation + temporary workspaces + kernel caches.
overhead_frac = 0.20
realistic_peak = backward_tensors * (1 + overhead_frac)
print("Realistic peak (+{}% overhead): {:.2f} GB".format(int(overhead_frac*100), realistic_peak))

gpu_gb = 40
print("\nFits in {} GB? {}".format(gpu_gb, realistic_peak < gpu_gb))

Input x:   12.58 GB
Output y:  12.58 GB
Weights W: 1.07 GB

Forward-only (optimistic): 26.24 GB
Backward tensors (still optimistic): 52.48 GB
Realistic peak (+20% overhead): 62.97 GB

Fits in 40 GB? False
